In [0]:
%python
# dml/06_carga_analytics_performance_fiis.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando consolidação dos TOP 3 FIIs de cada categoria para a tabela Gold...")

# %%
# 1. Monta a query unificada para capturar o TOP 3 de Tijolo, Papel e FoF
# Buscamos as notas e rankings calculados na camada Silver e unificamos tudo usando UNION ALL
qry_consolidacao_gold = f"""
  WITH top_tijolo AS (
    SELECT 
      s.ticker,
      'Tijolo' AS categoria_fii,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia
    FROM {catalogo}.{schema}.stg_scoring_tijolo s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking <= 3
  ),
  
  top_papel AS (
    SELECT 
      s.ticker,
      'Papel' AS categoria_fii,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia
    FROM {catalogo}.{schema}.stg_scoring_papel s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking <= 3
  ),
  
  top_fof AS (
    SELECT 
      s.ticker,
      'FoF' AS categoria_fii,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia
    FROM {catalogo}.{schema}.stg_scoring_fof s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking <= 3
  )
  
  -- Unifica os três resultados em uma única tabela de 9 registros
  SELECT * FROM top_tijolo
  UNION ALL
  SELECT * FROM top_papel
  UNION ALL
  SELECT * FROM top_fof
"""

df_gold = spark.sql(qry_consolidacao_gold)
df_gold.createOrReplaceTempView("temp_consolidacao_gold")

# %%
# 2. Executa a f-string de carga com INSERT OVERWRITE
tabela_destino = "analytics_performance_fiis"

qry_insert_dim_fiis = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino}
  SELECT 
    ticker,
    categoria_fii,
    nome_fundo,
    preco_atual,
    p_vp,
    dividend_yield_12m,
    score_final,
    posicao_ranking,
    data_referencia AS data_selecao,
    CURRENT_TIMESTAMP() AS data_carga
  FROM temp_consolidacao_gold
"""

print(f"Gravando a seleção final na tabela Gold: {catalogo}.{schema}.{tabela_destino}...")

# Grava de forma atômica limpando os dados anteriores da tabela Gold
spark.sql(qry_insert_dim_fiis)

print("✅ Tabela Gold 'analytics_performance_fiis' atualizada com SUCESSO com o TOP 9 FIIs do mercado!")